# ERC : fiabilité opérationnelle, risques et pré-sélection des FRU rebutées

Suite de `ERC_Aide_Decision_Composants.ipynb` : ce notebook en lit les sorties et ajoute trois couches à l'économie déjà calculée.

| Couche | Question | Source |
|---|---|---|
| Risque matière | La pièce contient-elle des matières critiques ou stratégiques (UE) ou des métaux précieux ? | Tables codées en dur (CRMA 2024/1252), composition qualitative par type |
| Risque supply | Manquer de cette pièce coûte-t-il cher (délai, source unique, fin de vie) ? | Table codée en dur, à remplir avec les achats |
| Fiabilité opérationnelle | Une pièce réparée ou récupérée tient-elle aussi bien qu'une neuve ? Combien de vie lui reste-t-il ? | `Parts_DI_ULS` (cache), a priori physiques, âge du donneur |

Puis la valeur nette par voie (vente, réemploi, remanufacturing) est tirée par Monte-Carlo sur des **plages** d'entrées, et la pré-sélection se fait par **front de Pareto** (aucune pondération imposée avant l'accord de l'équipe).

Règle : une entrée inconnue reste `None` et la sortie correspondante reste vide ; rien n'est remplacé par zéro.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJET = Path.cwd()
sys.path.insert(0, str(PROJET))
import erc_fiabilite_risques as efr

F_AIDE = PROJET / "ERC_Aide_Decision_Composants.xlsx"                       # sortie du notebook précédent
F_CACHE_ELF = PROJET / "Extraction_R26" / "fiabilite_ELF_references_ERC.csv"   # cache du fichier Parts_DI_ULS
F_POSES = PROJET / "Extraction_R26" / "poses_2022_2023_par_reference.csv"      # à extraire : pièces posées sur la période du fichier ELF
F_SORTIE = PROJET / "ERC_Fiabilite_Risques_Priorisation.xlsx"

FENETRE_ELF_JOURS = None      # durée de la fenêtre « early life failure » ; None = max observé (à confirmer avec l'auteur du fichier)
GARANTIE_JOURS = 365          # horizon de survie exigé d'une pièce réemployée (à fixer avec la qualité)
N_TOP = 30
pd.set_option("display.max_columns", 30)

aide = pd.read_excel(F_AIDE, sheet_name=None)
cl = aide["Classement_Tous_ateliers"]
fru_comp = aide.get("Composants_FRU_rebutees", pd.DataFrame())
print(f"{len(cl)} références classées | {len(fru_comp)} composants dans la nomenclature des FRU rebutées")

## 1. Risque matière

Présence des matières par type de pièce (IGBT, carte, condensateur, bobine, aimant…), reconnue sur la description. Score 0–1 : matière stratégique = 2 points, autre CRM = 1 point, plafonné à 8. **C'est une présence, pas une teneur** : la teneur vient de la déclaration matière du fabricant ou de l'analyse du recycleur. Le score sert deux voies : il augmente l'intérêt d'un recycleur certifié qui récupère ces matières, et il signale une exposition d'approvisionnement à long terme.

Les types non reconnus sont listés : compléter `efr.COMPOSITION_TYPE` plutôt que deviner.

In [ ]:
rm = pd.DataFrame([efr.risque_matiere(d) for d in cl["Description"]], index=cl.index)
rs = pd.DataFrame([efr.risque_supply(r, t) for r, t in zip(cl["Référence"].astype(str), rm["Type de pièce"])], index=cl.index)
cl = pd.concat([cl, rm, rs], axis=1)
print("Types de pièce :", cl["Type de pièce"].value_counts().to_dict())
print("Supply renseigné :", int(cl["Score risque supply (0-1)"].notna().sum()), "références sur", len(cl))
display(cl.loc[cl["Type de pièce"].eq("(type non reconnu)"), ["Référence", "Description", "Pièces"]].head(N_TOP))

## 2. Fiabilité opérationnelle

**Ce que contient vraiment le fichier `Parts_DI_ULS`.** Des défaillances, pas des survivants. Si ELF signifie *early life failure* (à confirmer avec l'auteur), chaque ligne est une pièce posée qui a lâché **dans une fenêtre** après la pose. Trois conséquences :

1. **Taux de défaillance précoce** : nombre d'ELF ÷ pièces posées sur la même période. Le dénominateur n'est pas dans le fichier : il faut les sorties de stock 2022-2023 par référence et par origine (neuve, `-R`, `-H`), à extraire en SQL. C'est l'indicateur clé du réemploi : **rapport du taux d'une pièce réparée ou récupérée à celui de la neuve**. Intervalles bayésiens (loi Bêta) pour ne pas sur-interpréter les petits effectifs.
2. **Forme de la défaillance** (Weibull `k`, vraisemblance tronquée à la fenêtre) : `k < 1` mortalité infantile, `k ≈ 1` aléatoire, `k > 1` usure. La forme dit **si l'âge d'une pièce récupérée est un problème** ; elle est estimable même sans dénominateur. L'échelle λ, elle, n'est pas identifiable avec des données tronquées : aucune durée de vie absolue n'est tirée de ce fichier seul.
3. **Durée de vie complète** : il faut des pièces encore en service (censure) : base installée avec dates de pose, ou retours terrain hors fenêtre ELF. À demander au service qualité / fiabilité terrain.

In [ ]:
elf = pd.read_csv(F_CACHE_ELF, sep=";", dtype=str, encoding="utf-8-sig") if F_CACHE_ELF.is_file() else pd.DataFrame()
poses = None
if F_POSES.is_file():                    # colonnes attendues : Référence, Origine (Neuve/Réparée/Récupérée), Pièces posées
    p = pd.read_csv(F_POSES, sep=";", encoding="utf-8-sig", dtype={"Référence": str})
    poses = p.set_index(["Référence", "Origine"])["Pièces posées"]
print("Lignes ELF :", len(elf), "| dénominateur des poses :", "chargé" if poses is not None else "absent → pas de taux, seulement la forme")

fiab = efr.fiabilite_par_reference(elf, FENETRE_ELF_JOURS, poses) if len(elf) else pd.DataFrame()
if len(fiab):
    fiab = fiab.sort_values("Défaillances ELF", ascending=False)
    print("Fenêtre utilisée (jours) :", FENETRE_ELF_JOURS or pd.to_numeric(elf["elf_days"], errors="coerce").max())
display(fiab.head(N_TOP))

In [ ]:
# Courbes de Kaplan-Meier par origine pour les références les plus défaillantes : une pièce réparée lâche-t-elle plus tôt ?
if len(elf):
    e = elf.assign(Base=elf["replacement_part_no"].map(efr.base_ref), Origine=elf["replacement_part_no"].map(efr.origine),
                   t=pd.to_numeric(elf["elf_days"], errors="coerce")).dropna(subset=["t"])
    for b in fiab["Référence"].drop_duplicates().head(5):
        for o, g in e[e["Base"].eq(b)].groupby("Origine"):
            km = efr.kaplan_meier(g["t"], np.ones(len(g)))
            q = {f"S≤{int(x)}j": km.loc[km["t"].le(x), "S(t)"].min() if km["t"].le(x).any() else 1.0 for x in (0, 7, 30, 90)}
            print(b, o, len(g), "défaillances | part restante de la cohorte défaillante :", {k: round(v, 2) for k, v in q.items()})

### A priori physiques (physics-of-failure)

Quand le terrain ne donne que la forme, la physique donne un ordre de grandeur. Deux lois standard, **à paramétrer avec la fiche technique de chaque référence** (rien n'est supposé ici) :

* **Condensateur électrolytique** (ex. 5953513, 450 V 4700 µF 105 °C) : la vie double tous les 10 °C sous la température nominale. Mécanisme d'usure (`k > 1`) : séchage de l'électrolyte. **Conséquence : candidat recyclage plutôt que réemploi**, sauf âge du donneur connu.
* **IGBT** (5477590, 5477591, 5173678) : fatigue en cyclage thermique, Coffin-Manson `Nf = A·ΔTj⁻ⁿ`, n ≈ 5 (LESIT). Un IGBT déposé *sain* lors d'une réparation où les deux sont changés n'a consommé qu'une partie de sa vie : c'est l'argument du réemploi après test.

In [ ]:
A_PRIORI = pd.DataFrame([
    # Référence, mécanisme, paramètre fiche technique, valeur (None = à lire sur la fiche), source
    ("5953513", "Électrolyte (Arrhenius 10 °C)", "Vie nominale (h) à 105 °C", None, "fiche fabricant"),
    ("5953513", "Électrolyte (Arrhenius 10 °C)", "Température de service (°C)", None, "mesure dans le SSSA ou conception"),
    ("5477590", "Cyclage thermique (Coffin-Manson)", "ΔTj typique (K)", None, "conception / fiche"),
    ("5477591", "Cyclage thermique (Coffin-Manson)", "ΔTj typique (K)", None, "conception / fiche"),
], columns=["Référence", "Mécanisme", "Paramètre", "Valeur", "Source"])
print("Ordre de grandeur, exemple de la règle seule (5000 h à 105 °C, service à 55 °C) :", efr.vie_condensateur_h(5000, 105, 55), "h")
print("Effet d'un ΔTj de 40 K au lieu de 60 K sur le nombre de cycles (rapport) :", round(efr.cycles_igbt(40) / efr.cycles_igbt(60), 1))
A_PRIORI

## 3. Durée de vie résiduelle d'un composant récupéré

Pour une pièce prise sur une FRU rebutée qui a servi `a` jours : probabilité de tenir la garantie `R(a+W)/R(a)` et vie résiduelle moyenne. Il faut donc **l'âge du donneur** : numéro de série de la FRU rebutée → date de pose (vwERT, ou base installée). Le tableau ci-dessous est **illustratif** (λ = 1 000 jours fixé arbitrairement) : il montre seulement comment la forme `k` change la décision.

In [ ]:
LAMBDA_ILLUSTRATIF = 1000
grille = pd.DataFrame([{"Forme k": k, "Lecture": efr.lecture_forme(k).split(" :")[0], "Âge du donneur (j)": a,
                        f"P(tenir {GARANTIE_JOURS} j)": efr.survie_conditionnelle(a, GARANTIE_JOURS, k, LAMBDA_ILLUSTRATIF),
                        "Vie résiduelle moyenne (j)": efr.vie_residuelle_moyenne(a, k, LAMBDA_ILLUSTRATIF)}
                       for k in (0.7, 1.0, 2.5) for a in (0, 180, 730, 1500)])
grille.round(2)

## 4. Valeur nette par voie (Monte-Carlo)

Chaque entrée accepte un nombre, une plage `(bas, haut)`, un triangle `(bas, mode, haut)` ou `None`. La sortie donne la moyenne, P5–P95 et **P(valeur > 0)** : une voie à forte moyenne mais P(>0) = 55 % ne vaut pas une voie modeste à 95 %.

* Vente : kg × prix recycleur − minutes de tri × taux horaire.
* Réemploi : rendement du test × (survie garantie × valeur de réemploi − (1 − survie) × coût d'une défaillance terrain) − (démontage + test) × taux horaire.
* Remanufacturing : rendement × valeur de réemploi − devis − transport − démontage.

Valeur de réemploi = MMICV × fraction (à définir par l'autre équipe) + bonus supply éventuel (coût d'une rupture évitée). Taux horaire = **heures en plus du plan de charge** (Stéphan, Lewis). Transport : à calculer depuis le registre acheteurs EMEA.

In [ ]:
# Une entrée par référence étudiée ; tout reste None tant que la source n'a pas répondu.
MODELE = dict(poids_kg=None, prix_recycleur_eur_kg=None, tri_min=None, taux_horaire_eur=None, mmicv=None, valeur_reemploi_frac=None,
              demontage_min=None, test_min=None, rendement_test=None, survie_garantie=None, cout_defaillance_terrain=None,
              devis_reman=None, transport_eur=None, rendement_reman=None, bonus_supply_eur=None)
HYPOTHESES = {}
for _, r in cl.head(N_TOP).iterrows():
    h = dict(MODELE)
    h["mmicv"] = r.get("MMICV unitaire (USD)") if pd.notna(r.get("MMICV unitaire (USD)")) else None   # devise à confirmer (USD/EUR)
    h["poids_kg"] = r.get("Poids unitaire (kg)") if pd.notna(r.get("Poids unitaire (kg)")) else None
    HYPOTHESES[str(r["Référence"])] = h
# Exemple de saisie quand les réponses arrivent :
# HYPOTHESES["5477590"] |= dict(prix_recycleur_eur_kg=(2, 6), taux_horaire_eur=(35, 45), demontage_min=(5, 10), test_min=(5, 15),
#                               rendement_test=(0.5, 0.9), valeur_reemploi_frac=(0.3, 0.5, 0.8), survie_garantie=(0.9, 0.99),
#                               cout_defaillance_terrain=(200, 800), tri_min=1)

lignes = []
for ref, h in HYPOTHESES.items():
    v, m = efr.valeur_voies(h), efr.manques_voies(h)
    for voie, s in v.items():
        lignes.append({"Référence": ref, "Voie": voie, **s, "Entrées manquantes": ", ".join(m[voie])})
voies = pd.DataFrame(lignes)
print("Voies calculables :", int(voies["moyenne"].notna().sum()), "sur", len(voies))
voies.head(N_TOP)

## 5. Pré-sélection : front de Pareto

Critères, tous « plus c'est haut, mieux c'est » : valeur en jeu (MMICV total), FRU rebutées qui contiennent la pièce, risque matière, risque supply (a priori de type tant que les achats n'ont pas répondu), et fiabilité du réemploi (inverse du rapport au neuf, quand le taux existe). Une référence **sur le front** n'est battue par aucune autre sur tous les critères à la fois : c'est la liste courte à instruire en premier. La note pondérée n'est donnée qu'à titre de comparaison, avec des poids égaux tant que l'équipe ne les a pas fixés.

In [ ]:
crit = cl[["Référence", "Description", "Pièces", "MMICV total (USD)", "Type de pièce", "CRM", "Métaux précieux",
           "Score risque matière (0-1)", "Score risque supply (0-1)", "A priori supply (type)"]].copy()
crit["Risque supply retenu"] = crit["Score risque supply (0-1)"].fillna(crit["A priori supply (type)"])
crit["Source supply retenue"] = np.where(crit["Score risque supply (0-1)"].notna(), "achats", np.where(crit["A priori supply (type)"].notna(), "a priori de type", "aucune"))
if len(fru_comp):
    crit["FRU rebutées (nomenclature)"] = crit["Référence"].astype(str).map(
        fru_comp.set_index(fru_comp["Référence"].astype(str))["FRU rebutées qui l'ont dans leur nomenclature"]).fillna(0)
else:
    crit["FRU rebutées (nomenclature)"] = np.nan
if len(fiab) and fiab["Rapport au neuf (taux)"].notna().any():
    r_ = fiab[fiab["Origine posée"].ne("Neuve")].groupby("Référence")["Rapport au neuf (taux)"].min()
    crit["Fiabilité réemploi (1/rapport)"] = 1 / crit["Référence"].astype(str).map(efr.base_ref).map(r_)
else:
    crit["Fiabilité réemploi (1/rapport)"] = np.nan
CRITERES = ["MMICV total (USD)", "FRU rebutées (nomenclature)", "Score risque matière (0-1)", "Risque supply retenu", "Fiabilité réemploi (1/rapport)"]
actifs = [c for c in CRITERES if crit[c].notna().any()]
crit["Sur le front de Pareto"] = efr.front_pareto(crit, actifs)
crit["Note (poids égaux, indicatif)"] = efr.score_pondere(crit, {c: 1 for c in actifs})
print("Critères actifs :", actifs, "| inactifs faute de données :", [c for c in CRITERES if c not in actifs])
print("Références sur le front :", int(crit["Sur le front de Pareto"].sum()))
crit.sort_values(["Sur le front de Pareto", "MMICV total (USD)"], ascending=False).head(N_TOP)

## 6. Ligne de tri et de démontage : coûts fixes, capacité, coûts variables par type de FRU

On ne décide plus pièce par pièce mais **pour une ligne** :

* **Probabilité qu'une pièce soit réutilisable, tirée de la consommation.** Si l'on consomme `c` pièces pour `N` FRU réparées qui en contiennent `n` chacune, un exemplaire est défectueux avec une probabilité ≈ `c / (N·n)`. Deux corrections, à calibrer avec l'audit de démontage : la **part préventive** (les deux IGBT changés quand un seul a lâché gonfle la consommation) et le **facteur rebut** (une FRU rebutée est plus abîmée qu'une FRU réparée).
* **Coûts fixes** : installation (postes, outillage ESD, banc de test) amortie + coûts fixes annuels (place, encadrement).
* **Capacité** : heures opérateur par an, en plus du plan de charge (Stéphan, Lewis).
* **Coûts variables par FRU** : temps d'**ouverture** propre au type de FRU (payé une fois) + temps d'**extraction et de test** de chaque pièce retenue, au **taux horaire du type de FRU**.

**Trois options pour chaque pièce de chaque FRU rebutée**, en valeur nette par FRU :

| Option | Valeur | Temps |
|---|---|---|
| 1 Démontage et réemploi | n × [p_réut × (valeur + délai évité) + (1 − p_réut) × valeur recyclage − transport] | ouverture soignée + extraction + test |
| 2 Tri et recyclage | n × kg × prix recycleur | ouverture sommaire + tri |
| 3 Jeter sans coût | 0 (référence actuelle) | aucun |

Une pièce qui échoue au test part au recyclage. La référence est la **situation actuelle** : FRU entière chez le recycleur actuel, au coût `cout_traitement_eur_kg` ; chaque kg sorti de sa benne évite ce coût. Formulation complète (ensembles, variables, fonctions de coût et de valorisation, objectif) : `Modele_Ligne_Demontage_Tri.md`. Le **délai d'approvisionnement** est le seul critère supply : chaque pièce réemployée évite `delai_semaines` d'attente, valorisées à `valeur_semaine_eur` (coût d'une semaine d'immobilisation × probabilité de rupture, à fixer avec les achats et le service).

L'optimisation choisit, pour chaque type de FRU, l'option de chaque pièce (ou jeter la FRU telle quelle) qui maximise la marge sous la contrainte d'heures. Elle donne aussi la **valeur nette d'une heure opérateur supplémentaire** : si elle est positive à pleine capacité, ajouter des heures rapporte.

In [ ]:
import erc_ligne_tri as elt

LIGNE = dict(cout_installation_eur=None, amortissement_ans=5, couts_fixes_annuels_eur=None,
             heures_operateur_an=None, taux_horaire_eur=None,
             cout_traitement_eur_kg=None)                                   # recycleur actuel : €/kg (0 si gratuit, négatif s'il paie)          # Stéphan / Lewis ; plages (bas, haut) acceptées
# Un type de FRU par entrée. p_reutilisable : via elt.p_defaut_depuis_conso puis elt.p_reutilisable.
FRU_TYPES = {
    "5453107-2-R": dict(rebut_an=None, poids_fru_kg=None, ouverture_min=None, ouverture_tri_min=None, taux_horaire_eur=None, pieces={
        p: dict(n_par_fru=None, extraction_min=None, test_min=None, p_reutilisable=None, valeur_eur=None, transport_eur=None,
                delai_semaines=None, valeur_semaine_eur=None,                      # réemploi + délai évité
                poids_kg=None, prix_recycleur_eur_kg=None, tri_min=None)            # recyclage
        for p in ("5477590", "5477591", "6850032", "5953513")
    }),
}
for f, d in FRU_TYPES.items():
    print(f, ": valeur des trois options par pièce (€/FRU, hors ouverture)")
    display(elt.valeurs_options(elt._milieu(d), LIGNE["taux_horaire_eur"]))
r = elt.point_milieu(LIGNE, FRU_TYPES)
display(r["detail"])
print("Situation actuelle (tout chez le recycleur actuel) contre ligne de démontage + tri :")
scenarios = elt.comparer_scenarios(LIGNE, FRU_TYPES)
display(scenarios)
pd.Series(r["resume"])

### Exemple fictif, pour lire les sorties

**Tous les chiffres de cette cellule sont inventés** : ils montrent le fonctionnement, pas l'ERC. Remplacer par les valeurs réelles dans la cellule précédente.

In [ ]:
ex_ligne = dict(cout_installation_eur=(20000, 40000), amortissement_ans=5, couts_fixes_annuels_eur=5000,
                heures_operateur_an=300, taux_horaire_eur=40, cout_traitement_eur_kg=(0.2, 0.5))
ex_frus = {
    "FRU A (type SSSA)": dict(rebut_an=400, poids_fru_kg=79, ouverture_min=(30, 45, 70), ouverture_tri_min=15, taux_horaire_eur=45, pieces={
        "IGBT x2": dict(n_par_fru=2, extraction_min=6, test_min=4, p_reutilisable=0.5, valeur_eur=30, delai_semaines=26, valeur_semaine_eur=1,
                        poids_kg=0.35, prix_recycleur_eur_kg=6, tri_min=1),
        "Carte driver": dict(n_par_fru=1, extraction_min=10, test_min=20, p_reutilisable=(0.2, 0.8), valeur_eur=300,
                             poids_kg=0.4, prix_recycleur_eur_kg=15, tri_min=2),
        "Condensateur x4": dict(n_par_fru=4, extraction_min=3, test_min=2, p_reutilisable=0.2, valeur_eur=10,
                                poids_kg=0.5, prix_recycleur_eur_kg=1.5, tri_min=0.5)}),
    "FRU B (type Smart Box)": dict(rebut_an=90, poids_fru_kg=5.5, ouverture_min=20, taux_horaire_eur=35, pieces={
        "Coque": dict(n_par_fru=1, extraction_min=5, test_min=2, p_reutilisable=0.7, valeur_eur=120),
        "Joystick": dict(n_par_fru=1, extraction_min=4, test_min=3, p_reutilisable=0.6, valeur_eur=55)}),
}
print("Trois options par pièce, FRU A (€/FRU, hors ouverture) :")
display(elt.valeurs_options(elt._milieu(ex_frus["FRU A (type SSSA)"]), 40).round(2))
ex = elt.point_milieu(ex_ligne, ex_frus)
display(ex["detail"].round(2))
display(pd.Series(ex["resume"]))
ex_sc = elt.comparer_scenarios(ex_ligne, ex_frus)
display(ex_sc.round(0)); print({k: round(v, 3) for k, v in ex_sc.attrs.items()})
mc = elt.monte_carlo_ligne(ex_ligne, ex_frus, n=300)
print("P(ligne rentable) :", round(mc["P(ligne rentable)"], 2), "| résultat P5 / médiane / P95 :", mc["Résultat P5 / médiane / P95 (€)"])
# Sensibilité à la capacité : où la ligne cesse-t-elle de gagner à avoir plus d'heures ?
sens = pd.DataFrame([{"Heures / an": h, **{k: v for k, v in elt.point_milieu(dict(ex_ligne, heures_operateur_an=h), ex_frus)["resume"].items()
                      if k in ("Résultat / an (€)", "Taux d'occupation", "Valeur d'une heure opérateur en plus (€ net)")}}
                     for h in (100, 200, 300, 500, 800)])
sens.round(2)

In [ ]:
lisez_moi = pd.DataFrame([
    ("Risque matière", "Présence qualitative par type de pièce ; listes CRM et stratégiques UE (Règlement 2024/1252) à re-vérifier. Pas de teneur."),
    ("Risque supply", "Délai d'approvisionnement seul (table SUPPLY, à remplir avec les achats) ; à défaut, a priori de type signalé comme tel. Dans la ligne de tri, le délai évité s'ajoute à la valeur du réemploi."),
    ("Fiabilité", "Fichier Parts_DI_ULS : défaillances seules. Taux = ELF / poses (dénominateur à extraire). Forme Weibull tronquée à la fenêtre ; λ non identifiable."),
    ("Vie résiduelle", "Exige l'âge du donneur (numéro de série → date de pose). Grille illustrative seulement."),
    ("Valeur par voie", "Monte-Carlo sur plages ; toute entrée None rend la voie non calculée. Estimations, distinctes des recettes réelles."),
    ("Ligne de tri", "Trois options par pièce : réemploi, recyclage, jeter sans coût. Coûts fixes amortis + capacité en heures + coûts variables par type de FRU (ouverture, extraction, test, taux horaire). p_réutilisable tiré de la consommation, corrigé par la part préventive et le facteur rebut."),
    ("Pré-sélection", "Front de Pareto sur les critères disposant de données ; note pondérée indicative à poids égaux."),
], columns=["Élément", "Contenu"])
with pd.ExcelWriter(F_SORTIE) as w:
    for n, d in {"Lisez_moi": lisez_moi, "Criteres_Pareto": crit, "Fiabilite": fiab, "A_priori_physiques": A_PRIORI,
                 "Vie_residuelle_illustr": grille, "Valeur_par_voie": voies,
                 "Options_par_piece": pd.concat([elt.valeurs_options(elt._milieu(d), LIGNE["taux_horaire_eur"]).assign(**{"Type de FRU": f}) for f, d in FRU_TYPES.items()]),
                 "Ligne_tri_detail": r["detail"], "Actuel_vs_ligne": scenarios, "Ligne_tri_resume": pd.DataFrame([r["resume"]])}.items():
        d.to_excel(w, sheet_name=n, index=False)
print("Écrit :", F_SORTIE.name)